**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Explore motif

```
MEME version 4

ALPHABET= ACGT

Background letter frequencies
A 0.25 C 0.25 G 0.25 T 0.25
```

In [2]:
TXT_FDIRY=${FD_DATA}/jaspar2024
TXT_FNAME="JASPAR2024_CORE_vertebrates_non-redundant.meme"
TXT_FPATH=${TXT_FDIRY}/${TXT_FNAME}

cat ${TXT_FPATH} | head -n 22

MEME version 4

ALPHABET= ACGT

strands: + -

Background letter frequencies
A 0.25 C 0.25 G 0.25 T 0.25

MOTIF MA0002.3 Runx1
letter-probability matrix: alength= 4 w= 9 nsites= 2000 E= 0
 0.061500  0.536000  0.074500  0.328000
 0.028500  0.000000  0.003500  0.968000
 0.000000  0.037500  0.936000  0.026500
 0.043500  0.063500  0.035000  0.858000
 0.000000  0.000000  0.993500  0.006500
 0.008500  0.021000  0.924000  0.046500
 0.005000  0.200000  0.125500  0.669500
 0.065500  0.231500  0.040500  0.662500
 0.250000  0.079000  0.144500  0.526500
URL http://jaspar.genereg.net/matrix/MA0002.3



## Export motif to numpy array

In [1]:
run_memelite python - <<'EOF'
import numpy as np
import os

from memelite.io import read_meme

# ====================================================================
# Import pwm_to_logodds from motifdelta
# --------------------------------------------------------------------
import sys
from pathlib import Path

FD_EXE = Path("/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts")
if str(FD_EXE) not in sys.path:
    sys.path.insert(0, str(FD_EXE))

from motifdelta import pwm_to_logodds

# ====================================================================
# Main function
# --------------------------------------------------------------------

def main(txt_fpath_inp, txt_fpath_out, txt_fpath_bg):
    """Main function"""

    ### Load empirical background
    arr_bg_B = np.load(txt_fpath_bg)
    print(f"Loaded background: {arr_bg_B}")

    ### Read MEME motifs
    dct_arr_motif_pwm_4xW = read_meme(txt_fpath_inp)
    print(f"Loaded {len(dct_arr_motif_pwm_4xW)} motifs from MEME file")

    ### Convert PWM to log-odds
    dct_arr_motif_pwm_Wx4 = dict()
    dct_arr_motif_lod_Wx4 = dict()

    for motif_name, arr_motif_pwm_4xW in dct_arr_motif_pwm_4xW.items():
        ### Convert pwm (4,W) -> pwm (W,4) -> log-odds (W,4)
        arr_pwm_Wx4 = arr_motif_pwm_4xW.T.astype(float)
        arr_lod_Wx4 = pwm_to_logodds(arr_pwm_Wx4, arr_bg_B)

        ### Collect results
        dct_arr_motif_pwm_Wx4[motif_name] = arr_pwm_Wx4
        dct_arr_motif_lod_Wx4[motif_name] = arr_lod_Wx4
    
    ### Export motif PWM and log-odds
    np.savez_compressed(
        txt_fpath_out,
        pwms  = np.array(dct_arr_motif_pwm_Wx4, dtype=object),
        lods  = np.array(dct_arr_motif_lod_Wx4, dtype=object),
        bg    = np.array(arr_bg_B, dtype=np.float32),
        names = np.array(list(dct_arr_motif_pwm_Wx4.keys()), dtype=object),
        alphabet="ACGT",
    )

    print(f"Saved {len(dct_arr_motif_pwm_Wx4)} motifs -> {txt_fpath_out}")
    
if __name__ == "__main__":

    ### Define input/output file path
    txt_fdiry_inp = "/hpc/group/igvf/kk319/data/jaspar2024"
    txt_fname_inp = "JASPAR2024_CORE_vertebrates_non-redundant.meme"
    txt_fpath_inp = os.path.join(txt_fdiry_inp, txt_fname_inp)

    txt_fdiry_out = "/hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard"
    txt_fname_out = "JASPAR2024_CORE_vertebrates_non-redundant.npz"
    txt_fpath_out = os.path.join(txt_fdiry_out, txt_fname_out)

    txt_fname_bg  = "background_zero_order.npy"
    txt_fpath_bg  = os.path.join(txt_fdiry_out, txt_fname_bg)
    
    ### Run main function
    main(txt_fpath_inp, txt_fpath_out, txt_fpath_bg)

EOF

Loaded background: [0.31302372 0.18711332 0.18779196 0.312071  ]
Loaded 879 motifs from MEME file
Saved 879 motifs -> /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/JASPAR2024_CORE_vertebrates_non-redundant.npz
